# Evaluation and export (S5.19/S5.21)

Runs the evaluation gates (schema/lint/gold/diagnosis thresholds), then exports GGUF Q4_K_M with SHA-256 manifests and model cards.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


**WARNING - THIS CELL STARTS MODEL EVALUATION.** loads the model and scores heldout plus Gold

In [ ]:
from colab_ux import gate, dry_run, warn_expensive

gate("evaluate", require_hf=False)
if dry_run():
    print("[DRY_RUN] MODEL EVALUATION not started")
else:
    warn_expensive("MODEL EVALUATION", "loads the model and scores heldout plus Gold")
    !PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/evaluate_model.py \n  --model "$HAVYN_DRIVE/checkpoints/standard" \n  --gold havyn-stage5/datasets/gold-v1 \n  --out "$HAVYN_DRIVE/evaluation"

**WARNING - THIS CELL STARTS GGUF EXPORT.** quantises and hashes the model; release gate must be PASS

In [ ]:
from colab_ux import gate, dry_run, warn_expensive

gate("live_export", require_hf=False)
if dry_run():
    print("[DRY_RUN] GGUF EXPORT not started")
else:
    warn_expensive("GGUF EXPORT", "quantises and hashes the model; release gate must be PASS")
    # Only meaningful when the gate report is PASS: export_gguf reads it
    # and marks NOT VERIFIED if any gate is not PASS.
    !PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/export_gguf.py \n  --model "$HAVYN_DRIVE/checkpoints/standard" --quant q4_k_m \n  --out "$HAVYN_DRIVE/artifacts" \n  --gate-report "$HAVYN_DRIVE/evaluation/gate-report.json"